# S03E03 — Reactor Robot Navigation

Agent prowadzi robota przez reaktor, unikając ruchomych bloków.
- Plansza 7×5, robot startuje z (1,5), cel to (7,5)
- Bloki (2 pola) poruszają się góra/dół cyklicznie
- Komendy: `start`, `right`, `left`, `wait`, `reset`

In [ ]:
import requests
import os
from dotenv import load_dotenv

load_dotenv("../.env")

API_KEY = os.getenv("AI_DEVS_API_KEY")
VERIFY_URL = "https://hub.ag3nts.org/verify"


def send_command(cmd: str) -> dict:
    """Wyslij komende do API reaktora."""
    resp = requests.post(VERIFY_URL, json={
        "apikey": API_KEY,
        "task": "reactor",
        "answer": {"command": cmd}
    })
    return resp.json()


def print_board(state: dict):
    """Wyswietl plansze w czytelnej formie."""
    for i, row in enumerate(state.get("board", [])):
        print(f"  Row {i+1}: {' '.join(row)}")
    p = state.get("player", {})
    print(f"  Robot: col={p.get('col')}, row={p.get('row')}")
    print()

In [ ]:
def predict_block_row5(block: dict) -> bool:
    """Czy blok bedzie na wierszu 5 po nastepnym ruchu?"""
    d = 1 if block["direction"] == "down" else -1
    return block["bottom_row"] + d >= 5


def col_blocked_now(blocks: list, col: int) -> bool:
    """Czy kolumna col (1-based) ma blok na wierszu 5 teraz?"""
    return any(b["col"] == col and b["bottom_row"] >= 5 for b in blocks)


def col_safe_after_move(blocks: list, col: int) -> bool:
    """Czy kolumna col (1-based) bedzie wolna w wierszu 5 po ruchu blokow?"""
    return not any(b["col"] == col and predict_block_row5(b) for b in blocks)


def choose_action(state: dict) -> str:
    """Wybierz akcje na podstawie stanu planszy.
    
    Priorytet: right > wait > left.
    Ruch w prawo jest bezpieczny gdy kolumna docelowa:
      - nie ma bloku na row 5 TERAZ
      - nie bedzie miala bloku na row 5 PO ruchu blokow
    """
    player_col = state["player"]["col"]
    blocks = state["blocks"]
    next_col = player_col + 1

    right_ok = (
        not col_blocked_now(blocks, next_col)
        and col_safe_after_move(blocks, next_col)
    )
    stay_ok = col_safe_after_move(blocks, player_col)

    if right_ok:
        return "right"
    elif stay_ok:
        return "wait"
    else:
        return "left"


print("Logika agenta gotowa.")

In [ ]:
MAX_STEPS = 100

state = send_command("start")
print(f"Code: {state.get('code')}, Message: {state.get('message')}")
print_board(state)

steps = 0
while steps < MAX_STEPS:
    # Cel osiagniety
    if state.get("reached_goal") or state.get("code") == 0:
        print(f"\nCEL OSIAGNIETY po {steps} krokach!")
        print(f"Message: {state.get('message')}")
        break

    # Blad (robot zginal) -> restart
    if state.get("code", 0) < 0:
        print(f"BLAD: {state.get('message')} -> restart")
        state = send_command("start")
        steps = 0
        print_board(state)
        continue

    # Brak danych gracza
    if "player" not in state:
        print(f"Nieoczekiwany stan: {state}")
        break

    action = choose_action(state)
    state = send_command(action)
    steps += 1

    p = state.get("player", {})
    print(f"Step {steps:2d}: {action:5s} -> col={p.get('col')} | code={state.get('code')}")

if steps >= MAX_STEPS:
    print(f"Przekroczono limit {MAX_STEPS} krokow.")

print(f"\nOdpowiedz: {state.get('message', '')}")